# GWAS Visualiser

This plots visualisations from the GWAS summary statistics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import os

In [ ]:
def plot_gwas(df_or_dfs, output_dir,
              title_prefix='GWAS',
              p_col='P',
              chr_col='chr',
              bp_col='position',
              beta_col='BETA',
              is_log10p=False,
              plot_range=None,        # [chr, startbp, endbp]
              gws_line=5e-8,
              suggestive_line=1e-5,
              suffix='',
              maf_filter=None,        # NEW: (low, high) to filter A1FREQ
              # New knobs for memory-efficiency:
              hist_bins=50,
              qq_max_points=100_000,  # reservoir sample size for Q-Q
              manhattan_downsample=None  # e.g., 2_000_000 to cap Manhattan points
             ):
    """
    Generates GWAS plots from a single DataFrame OR a list of per-chromosome DataFrames.

    Now also:
      - Optional maf_filter=(low, high): keep rows with A1FREQ in [low, high].
      - Appends "_{low}_{high}" to output filenames when maf_filter is used.
      - Q-Q plot on -log10(p) for observed vs expected.
      - Computes genomic inflation lambda (λ_GC) from the Q-Q reservoir sample (df=1)
        and annotates it on both the p-value histogram and the Q-Q plot.

    Memory-efficiency:
      - When given a list of per-chromosome DataFrames, data are processed chunk-by-chunk.
      - Manhattan: plotted incrementally; no global concat.
      - Histogram: bin counts are accumulated across chunks.
      - Q-Q: uses a reservoir sample (qq_max_points) across chunks.

    Args:
        df_or_dfs: pd.DataFrame or List[pd.DataFrame]
        output_dir (str): Where to save plots.
        title_prefix (str): For plot titles and filenames.
        p_col, chr_col, bp_col, beta_col: Column names.
        is_log10p (bool): True if p_col already stores -log10(P).
        plot_range (list/tuple): [chr, start_bp, end_bp] -> regional Manhattan only.
        gws_line (float): Genome-wide significance threshold (raw P).
        suggestive_line (float): Suggestive threshold (raw P).
        suffix (str): Suffix for filenames.
        maf_filter (tuple|list|None): (low, high) for filtering A1FREQ ∈ [low, high].
        hist_bins (int): Number of bins for P-value histogram.
        qq_max_points (int): Max points kept in RAM for Q-Q plot via reservoir sampling.
        manhattan_downsample (int|None): If set, cap total points plotted on Manhattan via sampling.

    Returns:
        - Single-DF mode: returns a cleaned DataFrame (like original).
        - List mode: returns a small summary dict (no giant concat kept in RAM).
    """

    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.stats import chi2  # NEW: for lambda_GC

    os.makedirs(output_dir, exist_ok=True)
    safe_title = title_prefix.replace(' ', '_').lower()

    # ----- maf filter parsing & filename tag -----
    maf_append = ''
    _maf_bounds = None
    if maf_filter is not None:
        try:
            lo, hi = [float(maf_filter[0]), float(maf_filter[1])]
            if lo > hi:
                lo, hi = hi, lo
            _maf_bounds = (lo, hi)
            maf_append = f'_{lo:g}_{hi:g}'
            print(f'Applying maf_filter: A1FREQ in [{lo}, {hi}]')
        except Exception:
            print('Warning: maf_filter is invalid; ignoring.')

    # ---------- helpers ----------
    def _ensure_raw_and_logp(df):
        """Drop NA, derive '-log10(P)' and '_raw_p' consistently; apply maf_filter on A1FREQ if requested."""
        df = df.dropna(subset=[p_col, chr_col, bp_col])
        # Apply maf filter first (to avoid unnecessary computations)
        if _maf_bounds is not None:
            lo, hi = _maf_bounds
            if 'A1FREQ' in df.columns:
                df = df.dropna(subset=['A1FREQ'])
                df = df[(df['A1FREQ'] >= lo) & (df['A1FREQ'] <= hi)]
            else:
                print("Warning: maf_filter specified but column 'A1FREQ' not found; skipping maf_filter.")

        if df.empty:
            return df

        if is_log10p:
            df = df[df[p_col] >= 0].copy()
            df['-log10(P)'] = df[p_col]
            df['_raw_p'] = np.power(10.0, -df[p_col])
        else:
            df = df[(df[p_col] > 0) & (df[p_col] <= 1)].copy()
            if df.empty:
                return df
            df['-log10(P)'] = -np.log10(df[p_col])
            df['_raw_p'] = df[p_col]
        return df

    def _chr_key(c):
        """Natural chromosome ordering key: 1..22 < X < Y < MT/M; fall back lexicographic."""
        if pd.isna(c):
            return (10**9, str(c))
        s = str(c).strip().upper()
        if s.startswith('CHR'):
            s = s[3:]
        alias = {'X': 23, 'Y': 24, 'MT': 25, 'M': 25, '0': 0}
        try:
            return (int(s), '')
        except ValueError:
            return (alias.get(s, 10**6), s)

    def _iter_chunks(df_or_dfs):
        """Yield (chr_name, df_chunk_prepared). Accepts single DF or list of DFs."""
        if isinstance(df_or_dfs, pd.DataFrame):
            pre = _ensure_raw_and_logp(df_or_dfs)
            if pre.empty:
                return
            for chr_name, sub in pre.groupby(chr_col, sort=False):
                yield chr_name, sub
        else:
            for sub in df_or_dfs:
                pre = _ensure_raw_and_logp(sub)
                if pre.empty:
                    continue
                for chr_name, sub2 in pre.groupby(chr_col, sort=False):
                    yield chr_name, sub2

    # --- regional-only fast path ---
    if plot_range is not None:
        try:
            zoom_chr, zoom_start, zoom_end = plot_range
        except Exception:
            print("Error: plot_range must be [chr, start_bp, end_bp].")
            return None

        print(f"Generating regional plot for CHR {zoom_chr}:{zoom_start}-{zoom_end}...")
        regional_df_list = []
        for chr_name, sub in _iter_chunks(df_or_dfs):
            def _norm(x):
                x = str(x).strip().upper()
                return x[3:] if x.startswith('CHR') else x
            if _norm(chr_name) == _norm(zoom_chr):
                regional_df_list.append(
                    sub[(sub[bp_col] >= zoom_start) & (sub[bp_col] <= zoom_end)]
                )

        if not regional_df_list:
            print("No data found in the specified range.")
            return None

        regional_df = pd.concat(regional_df_list, ignore_index=True)
        if regional_df.empty:
            print("No data found in the specified range.")
            return None

        plt.figure(figsize=(12, 6))
        plt.scatter(regional_df[bp_col], regional_df['-log10(P)'], s=20, alpha=0.7, rasterized=True)
        plt.axhline(y=-np.log10(gws_line), color='r', linestyle='--', label=f'GWS ({gws_line:.0e})')
        plt.axhline(y=-np.log10(suggestive_line), color='b', linestyle='--', label=f'Suggestive ({suggestive_line:.0e})')
        plt.xlabel(f'Position (Chromosome {zoom_chr})')
        plt.ylabel(r'$-log_{10}(P)$')
        plt.title(f'{title_prefix} - Regional Plot (CHR {zoom_chr}:{zoom_start}-{zoom_end})')
        plt.xlim(zoom_start, zoom_end)
        plt.legend()
        plt.tight_layout()
        plot_filename = f'{safe_title}_regional_plot_chr{zoom_chr}{maf_append}{suffix}.png'
        plt.savefig(os.path.join(output_dir, plot_filename), dpi=200)
        plt.close()
        print(f'Regional plot saved to {plot_filename}')
        return regional_df

    # ---------- FULL PLOTS ----------
    print(f'Generating full plots for "{title_prefix}" (memory-efficient mode enabled if list provided)...')

    # First pass: discover chromosomes, their max bp (for offsets), and total counts
    chr_lengths = {}
    chr_counts = {}
    chrom_set = set()
    total_points = 0

    for chr_name, sub in _iter_chunks(df_or_dfs):
        chrom_set.add(chr_name)
        max_bp = float(sub[bp_col].max()) if not sub.empty else 0.0
        chr_lengths[chr_name] = max(max_bp, chr_lengths.get(chr_name, 0.0))
        n_here = len(sub)
        chr_counts[chr_name] = chr_counts.get(chr_name, 0) + n_here
        total_points += n_here

    if not chrom_set:
        print("No valid P-values found after filtering. Aborting plot.")
        return None

    # Sort chromosomes naturally and compute cumulative offsets/centers
    chromosomes = sorted(list(chrom_set), key=_chr_key)
    offsets = {}
    centers = {}
    last_bp = 0.0
    for c in chromosomes:
        offsets[c] = last_bp
        centers[c] = last_bp + (chr_lengths.get(c, 0.0) / 2.0)
        last_bp += chr_lengths.get(c, 0.0)

    # --- Manhattan Plot (streamed) ---
    plt.figure(figsize=(15, 7))
    colors = ['#1f77b4', '#ff7f0e']
    plotted_points = 0
    ds_target = None if (manhattan_downsample is None) else int(manhattan_downsample)

    for i, c in enumerate(chromosomes):
        for chr_name, sub in _iter_chunks(df_or_dfs):
            if chr_name != c or sub.empty:
                continue

            x = sub[bp_col].to_numpy(dtype=float) + offsets[c]
            y = sub['-log10(P)'].to_numpy(dtype=float)

            if ds_target is not None and total_points > ds_target:
                k = max(1, int(len(sub) * ds_target / total_points))
                idx = np.random.default_rng().choice(len(sub), size=k, replace=False)
                x = x[idx]
                y = y[idx]

            plt.scatter(x, y, s=10, color=colors[i % len(colors)], alpha=0.7, rasterized=True)
            plotted_points += len(x)

    plt.axhline(y=-np.log10(gws_line), color='r', linestyle='--', label=f'GWS ({gws_line:.0e})')
    plt.axhline(y=-np.log10(suggestive_line), color='b', linestyle='--', label=f'Suggestive ({suggestive_line:.0e})')
    plt.xlabel('Cumulative BP')
    plt.ylabel(r'$-log_{10}(P)$')
    plt.title(f'{title_prefix} - Manhattan Plot')
    xticks_positions = [centers[c] for c in chromosomes]
    xticks_labels = [str(c) for c in chromosomes]
    plt.xticks(xticks_positions, xticks_labels, rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{safe_title}_manhattan_plot{maf_append}{suffix}.png'), dpi=200)
    plt.close()

    # --- Q-Q reservoir sample across chunks (also used for lambda_GC) ---
    rng = np.random.default_rng()
    sample = []
    n_seen = 0
    for _, sub in _iter_chunks(df_or_dfs):
        vals = sub['_raw_p'].to_numpy(dtype=float)
        m = len(vals)
        for j in range(m):
            n_seen += 1
            v = vals[j]
            if len(sample) < qq_max_points:
                sample.append(v)
            else:
                r = rng.integers(0, n_seen)
                if r < qq_max_points:
                    sample[r] = v

    # Compute lambda_GC from the sample (df=1). Uses Chi-square ISF so that small p -> large chi2.
    lambda_gc = np.nan
    if len(sample) > 0:
        sample_arr = np.asarray(sample, dtype=float)
        sample_arr = np.clip(sample_arr, np.finfo(float).tiny, 1.0)
        chisq = chi2.isf(sample_arr, df=1)
        lambda_gc = np.median(chisq) / chi2.ppf(0.5, df=1)  # denominator ≈ 0.454936...

    # --- P-value Histogram (with λ annotation) ---
    bin_edges = np.linspace(0, 1, int(hist_bins) + 1)
    bin_counts = np.zeros(len(bin_edges) - 1, dtype=np.int64)
    for _, sub in _iter_chunks(df_or_dfs):
        if sub.empty:
            continue
        counts, _ = np.histogram(sub['_raw_p'].to_numpy(dtype=float), bins=bin_edges)
        bin_counts += counts

    plt.figure(figsize=(8, 6))
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    widths = np.diff(bin_edges)
    plt.bar(bin_centers, bin_counts, width=widths, edgecolor='black', align='center')
    plt.xlabel('P-value')
    plt.ylabel('Frequency')
    plt.title(f'{title_prefix} - P-value Histogram')

    ax = plt.gca()
    text_hist = f'λ\u2093 (λ_GC) = {lambda_gc:.3f}\nN ≈ {total_points:,}\nQ-Q sample n = {len(sample):,}'
    ax.text(0.98, 0.98, text_hist, transform=ax.transAxes, ha='right', va='top',
            bbox=dict(facecolor='white', alpha=0.85, edgecolor='none'))

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{safe_title}_p_value_histogram{maf_append}{suffix}.png'), dpi=200)
    plt.close()

    # --- Q-Q Plot on -log10(p) (with λ annotation) ---
    if len(sample) == 0:
        print("No data for Q-Q plot.")
    else:
        observed_p = np.sort(np.asarray(sample, dtype=float))                 # ascending p
        n = observed_p.size
        expected_p = (np.arange(1, n + 1, dtype=float) - 0.5) / n

        obs_logp = -np.log10(observed_p)
        exp_logp = -np.log10(expected_p)

        plt.figure(figsize=(8, 8))
        plt.scatter(exp_logp, obs_logp, s=10, alpha=0.7, rasterized=True)

        maxval = float(np.nanmax([exp_logp.max(), obs_logp.max()]))
        plt.plot([0, maxval], [0, maxval], 'r--', linewidth=1)

        plt.xlabel(r'Expected $- \log_{10}(P)$')
        plt.ylabel(r'Observed $- \log_{10}(P)$')
        plt.title(f'{title_prefix} - Q-Q Plot (n={n:,} sampled of ~{total_points:,})')

        ax = plt.gca()
        text_qq = f'λ\u2093 (λ_GC) = {lambda_gc:.3f}'
        ax.text(0.02, 0.98, text_qq, transform=ax.transAxes, ha='left', va='top',
                bbox=dict(facecolor='white', alpha=0.85, edgecolor='none'))

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'{safe_title}_qq_plot{maf_append}{suffix}.png'), dpi=200)
        plt.close()

    print(f'Plotting completed successfully for "{title_prefix}".')

    # Return behavior mirrors original for single-DF input; in list-mode return a small summary
    if isinstance(df_or_dfs, pd.DataFrame):
        df_clean = df_or_dfs.drop(columns=['-log10(P)', '_raw_p', 'cumulative_bp'], errors='ignore')
        return df_clean
    else:
        return {
            'chromosomes': [str(c) for c in chromosomes],
            'n_variants': int(total_points),
            'qq_sample_size': int(len(sample)),
            'lambda_gc': float(lambda_gc) if np.isfinite(lambda_gc) else None,
            'manhattan_points_plotted': int(plotted_points),
        }


### Array-Based GWAS

In [ ]:
#Plot for the Array-based GWAS
array_summstats = pd.read_csv('../3_output/4_REGENIE/2_step2/array/hcm_step2_hcm.regenie', sep='\s+')
array_summstats.columns

In [ ]:
#Compute the mean chi-squared and the median chi-squared test statistics
print(f"Mean Chi-squared test statistic = {array_summstats['CHISQ'].mean(skipna=True)}")
print(f"Median Chi-squared test statistic = {array_summstats['CHISQ'].median(skipna=True)}")

In [ ]:
plot_gwas(array_summstats, '../3_output/5_GWAS_Manhattan_QQ/array/', 'HCM_GWAS_array','LOG10P', 'CHROM','GENPOS','BETA',is_log10p=True)

In [ ]:
#Plot for BAG3 locus
plot_gwas(array_summstats, '../3_output/5_GWAS_Manhattan_QQ/array/', 'HCM_GWAS_array','LOG10P', 'CHROM','GENPOS','BETA',is_log10p=True, 
          plot_range = [10,119651380,119677819], suffix='BAG3')

### srWGS ACAF-based GWAS

In [ ]:
#Import in the .regenie files
srwgs_acaf_regenie_files = [pd.read_csv(f'../3_output/4_REGENIE/2_step2/srWGS_ACAF/hcm_step2_chr{x}_hcm.regenie',sep='\s+') for x in range(1,23)]
# chr10_srwgs_acaf_regenie = pd.read_csv(f'../3_output/4_REGENIE/2_step2/srWGS_ACAF/hcm_step2_chr10_hcm.regenie',sep='\s+')

In [ ]:
plot_gwas(srwgs_acaf_regenie_files, '../3_output/5_GWAS_Manhattan_QQ/srWGS_ACAF/', 'HCM_GWAS_srWGS_ACAF','LOG10P', 'CHROM','GENPOS','BETA',is_log10p=True)

In [ ]:
#Add a MAF filter (0.01 to 0.99 A1FREQ) before plotting
plot_gwas(srwgs_acaf_regenie_files, '../3_output/5_GWAS_Manhattan_QQ/srWGS_ACAF/', 'HCM_GWAS_srWGS_ACAF','LOG10P', 'CHROM','GENPOS','BETA',
          is_log10p=True, maf_filter=(0.01,0.99))

In [ ]:
#Plot for BAG3 locus
plot_gwas(chr10_srwgs_acaf_regenie, '../3_output/5_GWAS_Manhattan_QQ/srWGS_ACAF/', 'HCM_GWAS_srWGS_ACAF','LOG10P', 'CHROM','GENPOS','BETA',True, 
          plot_range = [10,119651380,119677819], suffix='BAG3')